In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("retailcleaned.csv", header=None, low_memory=False)
df.columns = ['Invoice','StockCode','Description','Quantity','InvoiceDate','Price','Customer_ID','Country','Sourcesheet']

df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])
print(df.shape)
df.info()

(1033036, 10)
<class 'pandas.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 10 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  str           
 1   StockCode    1033036 non-null  str           
 2   Description  1028761 non-null  str           
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  str           
 5   Price        1033036 non-null  float64       
 6   Customer_ID  797885 non-null   float64       
 7   Country      1033036 non-null  str           
 8   Sourcesheet  1033036 non-null  str           
 9   invoiceDate  1033036 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(1), str(6)
memory usage: 164.7 MB


In [8]:
# FILTER AND CALCULATE RECENCY , FREQUENCY, MONETORY
rfm_data=df[(df['Customer_ID'].notna()) & (df['Quantity']>0)].copy()
snapshot_date = pd.to_datetime('2011-12-10')

rfm= rfm_data.groupby('Customer_ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', "nunique"),
    Monetary=('Quantity', lambda x: (x * rfm_data.loc[x.index, 'Price']).sum())
).reset_index()


In [11]:
rfm['R_score']=pd.qcut(rfm['Recency'],5,labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment_customer(row):
    if row['R_score']>=4 and row['F_score']>=4 and row['M_score']>=4:
        return 'Champion'
    elif row['R_score']>=3 and row['F_score']>=3:
        return 'Loyal'
    elif row['R_score']<=2 and row['F_score']>=3:
        return 'At Risk'
    elif row['R_score']>=4 and row['F_score']<=2 :
        return 'New'
    elif row['R_score']==3 and row['F_score']<=2 :
        return 'Needs Attention'
    else:
        return 'lost'

rfm['Segment']=rfm.apply(segment_customer,axis=1)
rfm['IsChurned']=rfm['Recency']>90
    

In [14]:
# MERGE RFM RESULTS BACK ONTO TRANSACTIONS AND EXPORT
df = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Revenue'] = df['Quantity'] * df['Price']

df_final = df.merge(
rfm[['Customer_ID', 'Recency', 'Frequency', 'Monetary', 'R_score', 'F_score', 'M_score', 'Segment', 'IsChurned']],
on='Customer_ID', how='left'
)
df_final.to_excel('EcommerceDashboardData.xlsx', index=False)